# Parse messages\nSplit text records into protocol-neutral message rows.

In [ ]:
project_root = "."
source = "data/capture"
pattern = "*.log*"
header = None
recursive = True
timezone = None
exclude_plugins = []
start = None
end = None
catalog = "rekep"
catalog_properties = {}
branch = "root"
target = "logs.messages"
static_values = {}
merge_by = True
batch_row_size = 65_536
commit_row_size = 250_000
limit = None

In [ ]:
from pathlib import Path

import pyarrow
import pyarrow.compute as pc
import pyarrow.fs
from rekep.filesystems import resolve
from rekep.iceberg import IcebergDataset
from rekep.text import TextFile, TextFiles
from rekep.times import unix_of
from rekep.urls import Url


def _location(value):
    parsed = Url.from_string(str(value))
    if parsed.scheme in {"", "file", "local"} and not Path(parsed.path).is_absolute():
        parsed = Url.from_path(project_root).join(parsed.path)
    return parsed.into_string()


lower, upper = unix_of(start), unix_of(end, upper=True)


def _bounded(batch):
    mask = None
    if lower is not None:
        mask = pc.greater_equal(batch.column("unix"), lower)
    if upper is not None:
        before = pc.less(batch.column("unix"), upper)
        mask = before if mask is None else pc.and_(mask, before)
    return batch if mask is None else batch.filter(mask)


declared = {
    "timezone": timezone,
    **({} if header is None else {"header_pattern": header}),
    "static_values": static_values,
}
location = _location(source)
filesystem, path = resolve(location)
info = filesystem.get_file_info(path)
if info.type == pyarrow.fs.FileType.NotFound:
    raise FileNotFoundError(location)
rows = (
    TextFiles.from_folder(location, pattern=pattern, recursive=recursive, **declared)
    if info.type == pyarrow.fs.FileType.Directory
    else TextFile.from_url(location, **declared)
)
field = rows.into_struct_field()

In [ ]:
messages = IcebergDataset(
    name=target,
    catalog=catalog,
    properties=dict(catalog_properties),
    branch=branch,
    field=field,
    commit_row_size=commit_row_size,
    sort_by=("unix", "hash"),
)

counts = {"read": 0}


def _batches():
    for batch in rows.read_arrow_reader(
        batch_row_size=batch_row_size, exclude_plugins=exclude_plugins
    ):
        batch = _bounded(batch)
        if limit is not None and counts["read"] + batch.num_rows > limit:
            batch = batch.slice(0, max(0, limit - counts["read"]))
        if batch.num_rows:
            counts["read"] += batch.num_rows
            yield batch
        if limit is not None and counts["read"] >= limit:
            break


written = messages.append_arrow_reader(
    _batches(), field, merge_by=merge_by, commit_row_size=commit_row_size
)
result = {
    "read": counts["read"],
    "written": written,
    "skipped": counts["read"] - written,
    "target": messages.name,
}
try:
    import scrapbook as sb
except ImportError:
    pass
else:
    sb.glue("result", result, encoder="json")
result